Install/import Torch 2.4.0 to match book examples

takes ~2.5 min on colab

In [4]:
#pip install torch==2.4.0

In [5]:
import torch
torch.__version__

'2.13.0+cu130'

In [6]:
# Check if NVIDIA GPU available
torch.cuda.is_available()

False

# A.2 Understanding Tensors

In [7]:
import torch
tensor0d = torch.tensor(1) # zero-dimensional tensor (scalar) from Python integer

tensor1d = torch.tensor([1, 2, 3]) # one dimensional tensor (vector) from python list

tensor2d = torch.tensor([[1, 2], [3, 4]]) # two-dimensional tensor from nested Python list

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]]) # three dimensional tensor from nested Python list

Tensor data types

In [8]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


In [9]:
# if we create tensors from Python floats, PyTorch creates tensors with a 32-bit precision by default
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [10]:
# 32-bit floating point offers balance between precision and computational efficiency
# GPU architectures optimized for 32-bit computations
# Possible to change precision using a tensor's .to method

# 64-bit integer tensor to 32-bit float tensor:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)


torch.float32


Common PyTorch tensor operations

In [11]:
# Create new tensors
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])
print(tensor2d)

tensor([[1, 2, 3],
        [4, 5, 6]])


In [12]:
# .shape allows access to shape of tensor
# output: torch.Size([2, 3]) = 2 rows, 3 columns
print(tensor2d.shape)

torch.Size([2, 3])


In [13]:
# To reshape tensor into a 3x2 tensor, can use .reshape method:
print(tensor2d.reshape(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [14]:
# HOWEVER, note that the more common command for reshaping tensors in PyTorch
# is .view()

print(tensor2d.view(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


subtle difference: .view() requires original data to be contiguous and will fail if not, reshape will work regardless

In [15]:
# Use .T to transpose a tensor (flip across its diagonal)
# NOT the same as reshaping
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [16]:
# common way to multiply two matrices is the .matmul method
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


In [17]:
# However, can also adopt the @ operation, which accomplishes the same thing more compactly
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


A.3 Seeing models as computation graphs

Logistic regression forward pass:

The model can be seen a sequence of intermediate computations (i.e. a computation graph)

In [18]:
import torch.nn.functional as F

y = torch.tensor([1.0]) # True label
x1 = torch.tensor([1.1]) # Input feature
w1 = torch.tensor([2.2]) # Weight parameter
b = torch.tensor([0.0]) # Bias unit
z = x1 * w1 + b # Net input
a = torch.sigmoid(z) # Activation & output
loss = F.binary_cross_entropy(a, y)

# A.4 Automatic differentiation made easy

The most common way of computing the loss gradients in a computation graph involves applying the chain rule (from calculus) from right to left, also called reverse-model automatic differentiation or backpropagation. We start from the output layer (or loss itself) and work backward through the network to the input layer. We do this to compute the gradient of the loss with respect to each parameter (weights and biases) in the network, which informs how we update these parameters during training.

We can obtain the partial derivative of the loss with respect to the trainable weight by chaining the individual partial derivative in the graph.

Similar to above, we can compute the partial derivative of the trainable derivative (bias??) by applying the chain rule.

Partial derivatives measure the rate at which a function changes with respect to one of its variables. A gradient is a vector containing all of the partial derivatives of a multivaraite function; a function with more than one variable as input.

In [19]:
# Computing gradients via autograd
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z  = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)





By default, PyTorch destroys the computation graph after
calculating the gradients to free memory. However, since we will reuse this computation graph shortly, we set retain_graph=True so that it stays in memory.

In [20]:
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

# The resulting values of the loss gradients given the model's parameters are:

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


Here we have been using the grad function manually, which can be useful for experimentation, debugging, and demonstrating concepts. But in practice, PyTorch provides even more high-level tools to automate this process, e.g. calling .backward() on the loss, which will compute the gradients of all leaf nodes in the graph, which will be stored via the tensors' .grad attributes:

In [21]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


# A.5 Implementing multilayer neural networks

The forward method describes how input data passes through the network and comes together as a computation graph. In contrast, the backward method, which we typically do not need to imlment ourselves, is used during training to compute gradients of the loss function given to the model parameters

(one example of an exception is using approximations for backward pass on LIF neurons since non-differentiable)

Note that Sequential class is not required, but easier if we have a series of layers to be executed sequentially: in the forward pass just need to call self.layers instead of calling each layer individually

In [24]:
# Multilayer perceptron with two hidden layers

class NeuralNetwork(torch.nn.Module):
  def __init__(self, num_inputs, num_outputs): # Coding the number of inputs and outputs as
                                               # vars allows reuse of same code for datasets
                                               # with different numbers of features and classes
      super().__init__()
      self.layers = torch.nn.Sequential(

         # 1st hidden layer
         torch.nn.Linear(num_inputs, 30), # Linear layer takes no. of input/output nodes as args
         torch.nn.ReLU(), # Nonlinear activ funcs place between hidden layers

         # 2nd hidden layer
         torch.nn.Linear(30, 20), # No. of output nodes of one hidden layer has to match no. of inputs to next layer
         torch.nn.ReLU(),

         # output layer
         torch.nn.Linear(20, num_outputs),
      )

  def forward(self, x):
        logits = self.layers(x)
        return logits # outputs of last layer are called logits

In [25]:
# We can then instantiate a new neural network object as follows:

model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Next, let's check total number of trainable parameters of this model:

In [26]:

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model paramteres:", num_params)

Total number of trainable model paramteres: 2213


These trainable parameters (i.e. each param for which requires_grad=True)
are contained, in our case, in the torch.nn.Linear layers (aka feedforward, fully connected)

From print(model), can see the first Linear layer is at index 0 in self.layers attribute. Can access corresponding weight parameter matrix as follows:

In [27]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.0451,  0.1185, -0.0572,  ...,  0.1092,  0.0855,  0.1252],
        [ 0.1335, -0.0963,  0.0925,  ..., -0.0335, -0.1069, -0.0207],
        [ 0.0704, -0.1262, -0.0003,  ...,  0.0475,  0.0139,  0.0938],
        ...,
        [-0.0689,  0.1043,  0.0316,  ...,  0.1246,  0.0600, -0.1357],
        [ 0.0997, -0.0203,  0.1020,  ...,  0.0580,  0.0232, -0.0145],
        [-0.0428, -0.1224,  0.0678,  ..., -0.1411,  0.0299, -0.1409]],
       requires_grad=True)


In [28]:
# since not shown in its entirety, use .shape to show its dimensions
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [29]:
# Similary, could also access bias vector via:
print(model.layers[0].bias)

Parameter containing:
tensor([-0.0481,  0.0807, -0.0004, -0.0505, -0.1054,  0.1166, -0.0059,  0.0137,
         0.1389, -0.0516, -0.0725,  0.0653, -0.0269,  0.0804,  0.1262, -0.0699,
        -0.0326,  0.1383,  0.0198, -0.0283, -0.0883, -0.0379, -0.0827, -0.0302,
        -0.0489,  0.0510,  0.0211,  0.1148, -0.1139, -0.0338],
       requires_grad=True)


Weights are initialized as small random numbers, can be made reproducible by seeding RNG with manual_seed:

In [30]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


Briefly demonstrate forward pass:

Generate single random training example X as a toy input (note that network expects 50=dimensional feature vectors) and feed to the model, generating three scores. Calling model(X) will automatically execute the forward pass of the model.

In [31]:
torch.manual_seed(123)
X = torch.rand((1,50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


Note that the output tensor also includes a grad_fn value. Here, grad_fn=<AddmmBackward0> represents last-used function to cmpute a variable in the computational graph. In particular, grad_fn=<AddmmBackward0> means that the tensor we are sinpecting was created via a matrix multiplication and addition (MAC?) operation. PyTorch will use this information when it cmoputes gradients during back-propagation. The <AddmmBackward0> part specifies operation performed; in this case an Addmm operation, which stands for matrix multiplication (mm) followed by an addition (Add).


If we just want to use a network without training or backprop (e.g. for predictions, inference after training) constructing this computational graph for backprop can be wasteful as it performs unncessary computations and consumes additional memory. So when using a model for inference (for instance, making predictions) rather than training, best practice is to use the torch.no_grad() context manager. This tells PyTorch that it doesn't need to keep track of the gradients, which can result in significant savings in memory and computation:

In [32]:
with torch.no_grad():
  out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In Pytorch, common practice to code models such that they return the outputs of the last layer (logits) without passing them to a nonlinear activation function. This is because PyTorch's commonly used loss functions combine the softmax (or sigmoid for biart classification) operation with the negative log-likelihood loss in a single class, the reason for which is numerical efficiency and stability.


So, if we want to compute class-membership probabilities for our predictions, we have to call the softmax function explicitly:


In [33]:
with torch.no_grad():
  out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


These values can now be interpreted as class-membership probabilities (summing up to 1). The values are roughly equal for this random input, which is expected fora randomly initialized model without training.

# A.6 Setting up efficient data loaders

1) We create a custom Dataset class that defines how individual data records are loaded

2) Using the Dataset class, we create different Dataset objects (e.g. test and train sets)

3) Each dataset is fed to/wrapped in a DataLoader.

4) Each DataLoader object handles dataset shuffling, assembling the data records into batches, and more



In [34]:
# Creating a small toy dataset

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, 1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([[-0.8, 2.8],
                       [2.6, -1.6],
                       ])
y_test = torch.tensor([0, 1])

# NOTE: PyTorch requires that class labels start with label 0, and
# the largest class label value should not exceed the number of output nodes
# minus 1 (since Python index starts at 0)
# e.g. if we have class labels 0, 1, 2, 3, and 4, the the output layer should
# consist of five nodes

In [35]:
# Defining a custom Dataset class

from torch.utils.data import Dataset

class ToyDataset (Dataset):
  def __init__(self, X, y):
    self.features = X
    self.labels = y

  # instructions for retrieving exactly one data record and the corresponding label
  def __getitem__(self, index):
    one_x = self.features[index]
    one_y = self.labels[index]
    return one_x, one_y

  # instructions for returning the total length of the dataset
  def __len__(self):
    return self.labels.shape[0]



train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

The purpose of this custom ToyDataset class is to instantiate a PyTorch DataLoader, but first, let's go over the general structure of the ToyDataset code:

The three main components of the a custom Dataset class in PyTorch are the \__init__ constructor, the \__getitem__ method, and the \__len__ method.

In the init method, we set up attributes that we can access later in the getitem and len methods. These could be file paths, file objects, database connectors, and so on. Since we created a tensor dataset that sits in memory, we simply assign x and y to these attributes, which are placeholders for our tensor objects.

In the getitem method, we define instructions for returning exactly one item from the dataset via an index. this refers to the features and the class label corresponding to a single traning example or test instance. (The dataloader will provide this index).

Finally, the len method contains instructions for retrieving he length of the dataset. Here, we use the .shape attribute of a tensor to return the number of rows in the feature array. In the case of the training dataset, we have five rows, which we can double check:



In [36]:
print(len(train_ds))

5


Now that we've defined a PyTorch Dataset class, we can use the DataLoader class to sample from it:

In [37]:
# Instantiating data loaders

from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds, # the ToyDataset instance created earlier serves as the input to the DataLoader
    batch_size=2,
    shuffle=True, # whether or not to shuffle the data
    num_workers=0 # no. of background processes
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False, # it is not necessary to shuffle a test dataset
    num_workers=0
)

In [38]:
# After instantiating the training data loader, we can iterate over it

# despite manual seeding, iterating multiple times will give different
# results due to shuffle=true
# this behavior is desired to prevent deep neural networks
# from getting caught in repetitive update cycles during training
for idx, (x, y) in enumerate(train_loader):
  print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[2.7000, 1.5000]]) tensor([1])


In [39]:
# test loader

for idx, (x, y) in enumerate(test_loader):
  print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.8000,  2.8000],
        [ 2.6000, -1.6000]]) tensor([0, 1])


we've specified a batch_size of 2, but the third batch only contains a single example, because we have 5 examples overall, which is not evenly divisible by 2.

In practice, having a substantially smaller batch as the last batch in a training epoch can distrub convergence during training. To prevent this, set drop_last=True, which will drop the last batch in each epoch:

In [40]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [41]:
# Now, iterating (enumerating?) over, we can see that last batch is dropped:
for idx, (x, y) in enumerate(train_loader):
   print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.5000,  2.6000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [ 2.3000, -1.1000]]) tensor([0, 1])


Lastly, on setting num_workers=0: this parameter is crucial for parallelizing data loading and preprocessing. When num_workers=0, data loading will be done in the main process and not separate worker processes. This may seem unproblematic, but can lead to significant slowdowns during training when we traing larger networks on a GPU. instead of focusing solely on processing the model, the PU must also take time to load and preprocess data, causing the GPU to sit idle.

However, not necessary in this case when working with very small datasets since total training time takes only fractions of as second anyways. In fact, overheard of spining up multiple worker processes (threads?) culd take longer than actual data loading when dataset is small.

Overall, beneficial when used correctly, but should be adapted to specific dataset size and computational environement for optimal results.

num_workers=4 often gives optimal performance on many real=world datasets, but depends on hardware and code used.

# A.7 A typical training loop

Let's now train a neural network on the toy dataset:

In [42]:
# Neural network training

import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2) # dataset has two features & two classes
optimizer = torch.optim.SGD(  # optimizer needs to know which parameters to optimize
    model.parameters(), lr=0.5
)

num_epochs = 3

# main loop
for epoch in range (num_epochs):
  model.train() # sets model to training mode

  for batch_idx, (features, labels) in enumerate(train_loader):
    logits = model(features)

    loss = F.cross_entropy(logits, labels)
    optimizer.zero_grad() # sets gradients from previous round to 0 to prevent unintended gradient accumulation
    loss.backward()
    optimizer.step()

    ### LOGGING
    print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
          f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
          f" | Train Loss: {loss:2f}")

  model.eval()
  # Insert optional model evaluation code

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.748736
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.645031
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.678278
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.259664
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.423583
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.024334


In [43]:
print("Total params:", sum(p.numel() for p in model.parameters()))

Total params: 752


In practice, often use third dataset - validation dataset - to find the optimal hyperparameter settings. Similar to test set, but while we only want ot use test set once to avoid biasing the evaluation, we usually use validation set multiple times to tweak the model settings.

model.train() and model.eval() are used to put the model into training and evaluation modes, which is necessary for components that behave differently during training and inference like dropout or batch normalization layers.

As discussed earlier, we pass logits directly into cross_entropy loss function, which will apply softmax internally. Then, calling loss.backward() will calc gradients in computation graph that PT constructed in the background. The optimizer.step() method will use the gradients to update the model parameters to minimize the loss. In the case of the SGD optimizer, this means multiplying the gradients with the larning rate and adding the scaled negative fradient to the paramters.

NOTE: to prevent undesired gradient accumulation, it is important to include an optimizer.zero_grad() call in each update round to reset the gradients to 0, otherwise gradients will accumulate, which may be undesired.

Having trained the model, we can now use it to make predictions:

In [44]:
model.eval()
with torch.no_grad():
  outputs = model(X_train)
print(outputs)

tensor([[ 2.9024, -3.8633],
        [ 2.4305, -3.2741],
        [ 1.7755, -2.4707],
        [-1.0086,  0.9308],
        [-0.7172,  0.4788]])


In [45]:
# To obtain class membership probabilities, feed raw logits into softmax:

torch.set_printoptions(sci_mode=False) # used to make outputs more legible
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9988, 0.0012],
        [0.9967, 0.0033],
        [0.9859, 0.0141],
        [0.1257, 0.8743],
        [0.2322, 0.7678]])


In [46]:
# We can convert these values into class label preictions use PT's argmax, which returns
# the index position of the highest value in each row if we set dim=1
# (setting dim=0 would return the highest value in each column instead)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [47]:
# Note that it is uncessary to compute softmax probailities to obtain the class labels.
# We could also apply the argmax function to the logits (outputs) directly:

preds = torch.argmax(outputs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


In [48]:
# Because dataset is relatively small, can compare to our true training labels by eye.
# Can also double check by using == comparison op
preds == y_train

tensor([True, True, True, True, True])

In [49]:
# using torch.sum, can count number of correct predictions
torch.sum(predictions == y_train)


tensor(5)

Since the dataset consists of five training examples, we have five out of five predictions that are correct, which has 5/5 x 100% = 100% prediction accuracy.

To generalize the computation of the prediction accuracy, let's implement a compute_accuracy func:

In [50]:
# A function to compute the prediction accuracy

def compute_accuracy(model, dataloader):
  model = model.eval()

  correct = 0.0
  total_examples = 0

  for idx, (features, labels) in enumerate(dataloader):

    with torch.no_grad():
      logits = model(features)

    predictions = torch.argmax(logits, dim=1)
    compare = labels == predictions # returns a tensor of T/F values depending on whether label matches
    correct += torch.sum(compare) # sum op counts no. of True values
    total_examples += len(compare)

  return (correct/total_examples).item()

This function iterates over a data loader to compute the number and fraction of correct predictions.

When working with large datasets, typically can only call the model on a small part of the dataset due to memory limitations. The compute_accuracy function is a ggeneral method that scales to datasets of arbitrary size since in each iteration, the dataset chunk that the model receives in the same size as the batch size seen during training.

We can then apply the function to the training:



In [51]:
print(compute_accuracy(model, train_loader))

1.0


In [52]:
# Similarly, we can apply the function to the test set:

print(compute_accuracy(model, test_loader))

1.0


# A.8 Saving and loading models

Now that we've trained our model, let's see how to save it so we can reuse it later.

Here is the recommended way of saving and loading models in PyTorch:

In [53]:
torch.save(model.state_dict(), "model.pth")

The model's state_dict is a Python dictionary object that maps each layer in the model to its trainable parameters (weights and biases). "model.pth" is an arbitrary filename for the model file saved to disk. We can give it any name and file ending we like; however, .pth and .pt are the most common conventions.

Once we have saved the model, we can restore it from disk:

In [54]:
model = NeuralNetwork(2,2)
model.load_state_dict(torch.load("model.pth"))



<All keys matched successfully>

The torch.load("model.pth") function reads the file "model.pth" and reconstructs the Python dictionary object containing the model's parameters while model.load_state_dict() applies these parameters to the model, effectively restoring its learned state from when we saved it.

The line model = Neuralnetwork(2,2) is not strictuly necessary if you execute this code in the same session where you saved a model. However, it is included here to illustrate that we need an instance of the model in memory to apply the saved parameters. Here, the NeuralNetwork(2, 2) architecture needs to match the original saved model exactly.

# A.9 Optimizing training performance with GPUs

In [55]:
# Double check that runtime supports GPU computing

print(torch.cuda.is_available())

False


In [56]:
# By default, tensor ops will be carried out on CPU
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


We can now use the .to() method. This method is the same as the one we use to change a tensor's datatype to transfer these tensors onto a GPU and perform the addition there:

In [57]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")
print(tensor_1 + tensor_2)

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

The resulting tensor now includes the device information, which indicates that the tensors reside on the first GPU. If your machine hosts multiple GPUs, you can specify which GPU you'd like to transfer the tensors to. You do so by indicateing the device ID in the transfer command. For instance, you can use .to("cuda:0") etc.

HOWEVER, all tensors must be on the same device. Otherwise, the computation will fail, where one tensor resides on the CPU and the other on the GPU:

In [ ]:
#tensor_1 = tensor_1.to("cpu")

print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='cuda:0')


# A.9.2 Single-GPU training

Now that we are familiar with transferring tensors to the GPU, we can modify the training loop to run on a GPU. THis steps requires only changing three lines of code:

In [ ]:
# A training loop on a GPU

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda") # define device var that defaults to gpu
model = model.to(device) # transfers model onto GPU

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

  model.train()
  for batch_idx, (features, labels) in enumerate(train_loader):
    features, labels = features.to(device), labels.to(device) # transfers data to GPU
    logits = model(features)
    loss = F.cross_entropy(logits, labels) # Loss function

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### LOGGING
    print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
          f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
          f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Insert optional model evaluation code

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.68
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.26
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.42
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.02


We can use .to("cuda") instead of device = torch.device("cuda") which works as well and is shorter. We can also modify the statement to make the same code executable on aCPU if a GPU is not available. This is considered best practice when sharing PyTorch code:




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# A.9.3 Training with multiple GPUs

(requires access to multiple GPUs)